# Introduction to Solving Mixed Integer Linear Programs (MILPs) in Python using PuLP 

Documentation:
- https://coin-or.github.io/pulp/

Install PuLP:
- https://coin-or.github.io/pulp/main/includeme.html#installation

```
python -m pip install pulp
```

Install other Free Solvers:
```
python -m pip install pulp[scip]
python -m pip install pulp[highs]
python -m pip install pulp[cylp]
```

These solvers require a license:
```
python -m pip install pulp[gurobi]   # free license for students
python -m pip install pulp[cplex]    # free license for students
python -m pip install pulp[xpress]
python -m pip install pulp[copt]
python -m pip install pulp[mosek]
```

---
## Example 1
A simple example from the documentation, where $x$ is a decision variable:
$$
\begin{flalign}
\min ~       & 4x_1 + x_{23} + y^{89}\\
\text{s.t.~} & x + y \le 2 \\
             & 0 \le x \le 3  \\
             & y \in \{0, 1\}   \\
             & \sum_{i = 1}^{n}
\end{flalign}
$$

```mermaid
flowchart TD
A[Deploy] --> B[What?]
```

In [40]:
from pulp import *

In [41]:
# Create a problem object:
prob = LpProblem("myProblem", LpMinimize)

# Define decision variables:
x = LpVariable("x", 0, 3)
y = LpVariable("y", cat="Binary")

# Define objective [Note: no (in)equality here]:
prob += -4*x + y

# Define a constraint [Note: There IS an inequality here]:
prob += x + y <= 2

In [42]:
# Solve problem with default solver:
status = prob.solve()

# Alternatively, could use other solvers:
# status = prob.solve(GLPK(msg = 0))

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/cmurray3/venvs/vrv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/ws/02f4jv7x1z52f9ms1wv8518h004g86/T/aa646c4be99a4d50961efe1393cab00a-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/ws/02f4jv7x1z52f9ms1wv8518h004g86/T/aa646c4be99a4d50961efe1393cab00a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 6 COLUMNS
At line 13 RHS
At line 15 BOUNDS
At line 18 ENDATA
Problem MODEL has 1 rows, 2 columns and 2 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -8 - 0.00 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from -8 to -1.79769e+308
Probing was tried 0 times and created 0 cuts of which 0 were

In [43]:
# Display status (e.g., 'Optimal', 'Unbounded', 'Infeasible'):
LpStatus[status]

'Optimal'

In [44]:
# Get the solution:
value(x), value(y)

(2.0, 0.0)

In [45]:
# Get the objective function value:
print(f"Objective function value = {value(prob.objective)}")

Objective function value = -8.0


## Example 2 - Instance of a Knapsack Problem

$$
\begin{flalign}
\text{Max } & 60x + 100y + 120z \\
\text{s.t. } & 10x + 20y + 30z \le 50 \\ 
& x, y, z \in \{0, 1\}
\end{flalign}
$$

In [ ]:
FIXME -- You do this

## Example 3 - General Knapsack Formulation
- There are $n$ products/items
- Each item $i \in \{1, 2, \ldots, n\}$ has:
    - a reward, $r_i$;
    - a weight, $w_i$; and
    - a binary decision, $x_i$, regarding whether to select this item.
- There is an overall weight limit, $w^{\max}$
- The objective is to maximize the total reward, subject to not violating the maximum weight limit/constraint, by deciding which items to include in the collection.

The binary LP is given by:
$$
\begin{flalign}
\text{Max } & \sum_{i = 1}^{n} r_i x_i \\
\text{s.t. } & \sum_{i = 1}^{n} w_i x_i \le w^{\max} \\
& x_i \in \{0, 1\} \quad \forall ~ i \in \{1, 2, \ldots, n\}
\end{flalign}
$$

In [46]:
# Let's create an example instance, where our items are books.

# 1. How many books are available?
n = 4

# 2. What are the rewards of our books?
rewards = {1: 70, 2: 100, 3: 120, 4: 80}

# 3. What are the weight of our books?
weights = {1: 15, 2: 25, 3: 35, 4: 20}

# 4. What's the max weight we can carry?
wmax = 65

In [48]:
# Create the 'prob' variable to contain the problem data
prob = LpProblem("General Knapsack", LpMaximize)

items = range(1, n+1)

# A dictionary called 'item_vars' is created to contain the decision variables
item_vars = LpVariable.dicts("Item", items, cat="Binary") 

# Add the objective function to `prob`:
prob += (
    lpSum([rewards[i] * item_vars[i] for i in items]),
    "Total_reward_of_selected_items",
)

# Add the knapsack constraint:
prob += (
    lpSum([weights[i] * item_vars[i] for i in items]) <= wmax,
    "Weight_Capacity_Limit",
)

# Write the problem data to an `.lp` file (optional, but helpful for debugging):
prob.writeLP("knapsackModel.lp")

[Item_1, Item_2, Item_3, Item_4]

In [49]:
!cat knapsackModel.lp

\* General_Knapsack *\
Maximize
Total_reward_of_selected_items: 70 Item_1 + 100 Item_2 + 120 Item_3
 + 80 Item_4
Subject To
Weight_Capacity_Limit: 15 Item_1 + 25 Item_2 + 35 Item_3 + 20 Item_4 <= 65
Binaries
Item_1
Item_2
Item_3
Item_4
End


In [50]:
# Solve the problem using PuLP's choice of solver:
prob.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/cmurray3/venvs/vrv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/ws/02f4jv7x1z52f9ms1wv8518h004g86/T/e01bfdf492284637b90e7809011ad269-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/ws/02f4jv7x1z52f9ms1wv8518h004g86/T/e01bfdf492284637b90e7809011ad269-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 6 COLUMNS
At line 23 RHS
At line 25 BOUNDS
At line 30 ENDATA
Problem MODEL has 1 rows, 4 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 267.143 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 1 strengthened rows, 0 substitutions
Cgl0004I processed model has 1 rows, 4 columns (4 integer (4 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 0 integer

1

In [51]:
# Print the status of the solution
print("Status:", LpStatus[prob.status])

Status: Optimal


In [52]:
# Print the objective function value:
print(f"Objective function value = {value(prob.objective)}")

Objective function value = 250.0


In [53]:
# Print the decision variables and their values
# (but only show the non-zero ones):
for v in prob.variables():
    if (v.varValue != 0):
        print(f"{v.name} = {v.varValue}")

Item_1 = 1.0
Item_2 = 1.0
Item_4 = 1.0


---
## Excercise 1 - Solve a Knapsack Problem from .csv Data
Use our knapsack formulation on the data in the `knapsack_data.csv` file.

In [54]:
!cat knapsack_data.csv

ID,Value,Weight,Category
73328137,161,32,1
43113515,201,24,1
24109828,52,9,1
51096300,233,9,2
39519934,248,9,2
79292226,81,16,1
36886541,189,32,2
72299481,103,30,2
22368599,185,27,1
32802902,112,14,2
58788014,185,29,2
52148656,207,14,1
81954510,121,7,1
30904470,197,26,1
37415187,106,5,1
28525302,124,9,2
36212161,213,22,1
44582451,112,41,2
32607029,113,36,2
49068620,123,12,2
97243323,160,36,1
59968491,65,48,2
29050922,120,38,2
74149058,125,35,1
33043198,180,49,1
48409717,230,26,1
92429492,215,42,1
26163155,228,20,1
65674890,74,30,1
77704478,63,5,1